# 2. In-context changes

This notebook shows the **context intervention** on the original 22-fraud-case dataset. It checks the source split, selects random and balanced labeled examples at a fixed 128-row budget, and verifies that held-out target labels are masked in SAP requests. SAP RPT model weights stay fixed. The [before notebook](01_before_context_engineering.ipynb) reports the original result; the [after notebook](03_after_context_engineering.ipynb) evaluates the saved predictions.

## Setup

The first 1,024 rows are the eligible pool. The final 128 invoices, including 22 fraud cases, are held out. For each target, seed 11 selects 128 random or balanced labeled rows. Selection and request validation happen here; measured outcomes appear only in the after notebook.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
FIONEER_BLUE = '#0F37FF'
FIONEER_NAVY = '#353C54'
FIONEER_LIGHT_BLUE = '#CFDEFF'
FIONEER_YELLOW = '#FFD53C'
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'archive/synthetic_invoice_experiments/published_baseline/payment_behavior.csv').exists())
sys.path[:0] = [str(ROOT / 'src'), str(ROOT / 'scripts')]
from context_engineering import assert_no_leakage, select_context
from run_context_benchmark import request_for, metric_row
source_path = ROOT / 'archive/synthetic_invoice_experiments/published_baseline/payment_behavior.csv'
source = pd.read_csv(source_path)
pool, holdout = source.iloc[:1024].copy(), source.iloc[1024:].copy()
targets = ('paid_late', 'is_fraud')
seed, budget = 11, 128
assert len(source) == 1152 and len(pool) == 1024 and len(holdout) == 128
assert int(holdout.is_fraud.sum()) == 22 and int(holdout.paid_late.sum()) == 51
assert source.invoice_id.is_unique and not source.isna().any().any()
assert_no_leakage(pool, holdout)


## Step 1: check the data and split
The older eight-case context files are a separate pilot and are not used here. Check IDs, missing values, and target counts before looking at model results.

In [2]:
audit = pd.DataFrame({'slice': ['eligible pool', 'final test'],
                      'rows': [len(pool), len(holdout)],
                      'late_payment_cases': [int(pool.paid_late.sum()), int(holdout.paid_late.sum())],
                      'fraud_cases': [int(pool.is_fraud.sum()), int(holdout.is_fraud.sum())]})
display(audit)
assert set(pool.invoice_id).isdisjoint(holdout.invoice_id)


,slice,rows,late_payment_cases,fraud_cases
0,eligible pool,1024,450,187
1,final test,128,51,22


## Step 2: select the context rows
The **random** condition samples 128 unique labeled rows. The **balanced** condition samples as close to half positive and half negative as the eligible pool permits, separately for each target. Both use the same seed and 128-row budget. This is an explicit selection intervention, not a change to the fraud labels. The earlier eight-case pilot suggested this strategy; its scores are not reused here. This 22-case run is exploratory because strategy choice was informed by that related pilot.

In [3]:
contexts = {}
composition = []
for target in targets:
    for strategy in ('random', 'balanced'):
        selected, _ = select_context(pool, target, holdout.iloc[:1], strategy, budget, seed)
        assert_no_leakage(pool, holdout, selected)
        contexts[(target, strategy)] = selected
        composition.append({'target': target, 'selection': strategy,
                            'rows': len(selected), 'positive_examples': int(selected[target].sum()),
                            'negative_examples': int((selected[target] == 0).sum())})
display(pd.DataFrame(composition))


,target,selection,rows,positive_examples,negative_examples
0,paid_late,random,128,51,77
1,paid_late,balanced,128,64,64
2,is_fraud,random,128,27,101
3,is_fraud,balanced,128,64,64


## Step 3: build a masked SAP prediction request
Only the selected labeled examples supply the target. All final-test target cells are `[PREDICT]`. The other outcome is excluded because it would be unavailable when predicting a new invoice. The invoice ID is an index, not a predictor.

In [4]:
sample = request_for(contexts[('is_fraud', 'balanced')], holdout, 'is_fraud')
assert sample['index_column'] == 'invoice_id'
assert len(sample['rows']) == budget + len(holdout)
assert all(row['is_fraud'] == '[PREDICT]' for row in sample['rows'][budget:])
assert all('paid_late' not in row for row in sample['rows'])
display(pd.DataFrame([{'context_rows': budget, 'prediction_rows': len(holdout),
                       'target': sample['prediction_config']['target_columns'][0]['name'],
                       'query_targets_masked': True, 'other_target_excluded': True}]))


,context_rows,prediction_rows,target,query_targets_masked,other_target_excluded
0,128,128,is_fraud,True,True


## What changed

The two new SAP RPT conditions use the same target, 128-row context budget, selection seed, request format, and held-out invoices. Only the example-selection strategy changes. The original first benchmark used 1,024 labeled rows and a different request schema, so the matched random-versus-balanced pair is the comparison that isolates selection. The [after notebook](03_after_context_engineering.ipynb) checks its saved prediction rows and reports the measured tradeoff.